# Python File Handling: `os` vs `pathlib` (Corporate Standards)

## Why Modern Companies Prefer `pathlib`
Introduced in Python 3.4, `pathlib` has largely replaced `os.path` in modern enterprise codebases. 
*   **Object-Oriented**: It treats paths as smart objects instead of plain text strings.
*   **Cross-Platform**: It automatically manages slash directions (`/` for Mac/Linux and `\` for Windows).
*   **Chaining Syntax**: It reduces code length by allowing you to chain actions together.

## When Companies Still Use `os`
*   **Legacy Code**: Maintaining older infrastructure built before Python 3.4.
*   **System Environment Operations**: Accessing system-level values like `os.environ` or `os.name` which `pathlib` cannot do.


# Data Engineering Notes: Mastering `pathlib`

## The Data Pipeline Problem
Every **ETL (Extract, Transform, Load) pipeline** starts with locating files. Common production tasks include:
*   *"Read all CSV files from this folder."*
*   *"Find all JSON configuration files."*
*   *"Create today's report inside the Reports folder."*
*   *"Move failed files into Archive."*

### Typical Pipeline Architecture
```text
Data Folder ➔ CSV / JSON / Excel ➔ Process ➔ Output Report
```
To build professional pipelines, you must handle paths cleanly. This is why **`pathlib`** is a foundational topic.

---

## What is `pathlib`?
Many beginners think `pathlib` is just another way to open files, but it is much more powerful. It is Python's modern, **object-oriented** library for interacting with file systems.

*   **Old String Style**: `path = "data/report.csv"` (Treats paths as basic text strings)
*   **Modern Object Style**: `path = Path("data/report.csv")` (Treats paths as intelligent objects)

### Why Python Replaced `os.path`
Before Python 3.4, developers used fragmented functions from the `os` module:
*   `os.path.basename(...)` to get the filename
*   `os.path.dirname(...)` to get the parent folder
*   `os.path.splitext(...)` to separate extensions

`pathlib` unifies all of these capabilities into a single, cohesive **Path Object**:
```text
Path Object
 │
 ├── Properties: .name, .suffix, .parent
 └── Methods:    .exists(), .mkdir(), .glob(), .is_file(), .is_dir(), .open()
```

---

## Why Strings Fail in Production
When you define a path as a raw string like `path = "input/employees.csv"`, Python only views it as a sequence of **characters**. It cannot determine:
*   If the path physically exists
*   Whether it points to a file or a directory
*   What its parent directory or extension is

When wrapped in `Path("input/employees.csv")`, Python converts it into a structural **File System Object** that actively knows its location, metadata, and properties.

### Internal Mechanics
```text
"input/employees.csv" ➔ Path() ➔ Path Object Stores: [Directory, Filename, Extension, Absolute Location]
```

---

## Real-World Production Context

### Target Project Architecture
```text
Project
│
├── config/ ── config.json
├── input/  ── employees.csv, commits.csv, builds.csv
├── output/ ── report.csv
└── logs/   ── app.log
```

### Batch Processing Use Case
Consider an engineering reports directory organized by month:
```text
Engineering Reports
│
├── Jan/ ── report.csv
├── Feb/ ── report.csv
├── Mar/ ── report.csv
└── Summary/
```
Using `pathlib`, scanning and processing every `report.csv` file across these folders becomes clean and straightforward using built-in directory traversal methods like `.glob()`.


## Internal Working

"employees.csv"

↓

Path()

↓

Create Path Object

↓

Stores

• Path components
• File name
• Parent directory
• Extension
• File system operations

↓

Provides methods such as

exists()
is_file()
is_dir()
mkdir()
glob()
rename()

# Importing pathlib

The `pathlib` module is built into Python.

Import it using:

```python
from pathlib import Path
```

No installation is required.

## Creating Your First Path Object

In [2]:
from pathlib import Path
path=Path("employees.csv")

print(path)
print(type(path))

employees.csv
<class 'pathlib._local.WindowsPath'>


### Deep Dive: Path Object Initialization Anatomy

```python
from pathlib import Path

path = Path("employees.csv")
```

When you execute the line above, Python performs an explicit 4-step initialization process to transform a raw string into a smart file system object:

```text
  [ "employees.csv" ]  <-- Step 1: Raw String Input
          │
          ▼
   Path Constructor    <-- Step 2: Instantiation Call
          │
          ▼
  [  Path Object   ]   <-- Step 3: Metadata Layout Stored (Lazy Evaluation)
          │
          ▼
   Variable: path      <-- Step 4: Reference Assignment
```

#### Step-by-Step Breakdown

*   **Step 1: String Parsing**: Python reads the argument `"employees.csv"` as a primitive, sequential string of characters.
*   **Step 2: Constructor Processing**: The string is passed directly into the `Path()` class constructor method.
*   **Step 3: Object Generation (Lazy Evaluation)**: Python initializes a unique `Path` object inside memory. It extracts and stores foundational parameters (structural properties like name, stem, and suffix hints). Crucially, **it does not touch the hard drive** or check if the file physically exists yet; this keeps the creation process incredibly lightweight.
*   **Step 4: Variable Assignment**: The fully instantiated object pointer is stored in the variable `path`, ready to execute any future file system actions.


One of the single biggest reasons software engineering and data teams mandate `pathlib` is to eliminate **Operating System (OS) compatibility bugs** in production code. 

As a beginner, understanding how `pathlib` manages this transitions you from writing "local script" code to "production-grade" engineering code.

---

## 1. The Core Conflict: Windows vs. Linux/Mac
Operating systems use completely different characters to separate folders in a path string:
*   **Windows**: Uses the backslash (`\`), e.g., `"data\input\employees.csv"`
*   **Linux / Mac**: Uses the forward slash (`/`), e.g., `"data/input/employees.csv"`

### The Danger of Raw Strings
If you hardcode a path using normal string characters on your Windows laptop, your code will **crash immediately** the moment it is deployed onto a Linux-based cloud server (like AWS or Docker containers), because Linux does not recognize the backslash as a folder separator.

---

## 2. How `pathlib` Quietly Solves This (The `/` Operator)
Instead of forcing you to guess which slash to use, `pathlib` overrides Python's division operator (`/`) and turns it into a universal **Path Combiner**. 

When you write paths using this operator, your code becomes completely decoupled from string characters:

```python
from pathlib import Path

# The safe, universal corporate standard
clean_path = Path("data") / "input" / "employees.csv"
```

---

## 3. Under the Hood: `WindowsPath` vs. `PosixPath`
When you instantiate a `Path` object, Python looks at your machine's operating system at runtime and automatically chooses a specialized sub-class behind the scenes:

### Scenario A: Running on your Windows Laptop
1. Python detects the Windows OS architecture.
2. It quietly turns your `Path` object into a **`WindowsPath`** object.
3. When interacting with your hard drive, it automatically translates everything to use backslashes (`\`).

### Scenario B: Running on a Colleague's Linux System (or Cloud Server)
1. Python detects the Linux/Unix (POSIX) architecture.
2. It quietly turns that exact same line of code into a **`PosixPath`** object.
3. When interacting with the storage engine, it automatically translates everything to use forward slashes (`/`).

**The Ultimate Rule**: You write the code exactly once. `pathlib` dynamically reshapes the paths to match whatever machine is executing the file.
Use code with caution.Cell: Codepythonfrom pathlib import Path
import os

# Create a path using the universal combining slash operator
notebook_path = Path("workspace") / "projects" / "etl_pipeline"
```python
print("==================================================")
print("       CROSS-PLATFORM PATH VERIFICATION           ")
print("==================================================")
print(f"1. Your current OS architecture type: {os.name}")
print("   ('nt' means Windows, 'posix' means Mac or Linux)")
print("--------------------------------------------------")
print(f"2. Active Class Type generated by Python: {type(notebook_path).__name__}")
print("--------------------------------------------------")
print(f"3. How this system renders your file path: {notebook_path}")
print("==================================================")
```
Use code with caution.Would you like to move on to the next major pathlib topic: how to automatically scan a directory for specific files using .glob()?

# Relative Path vs Absolute Path

Every file and folder on a computer has a location.

Python can represent this location in two ways:

1. Relative Path
2. Absolute Path
# Understanding Paths: Absolute vs. Relative

When working in data engineering pipelines, locating your datasets correctly depends entirely on understanding the two types of file paths: **Absolute Paths** and **Relative Paths**.

---

## 1. Absolute Paths (The Full Address)
An **Absolute Path** is the complete, unambiguous location of a file or folder starting from the absolute root of the file system.

*   **Windows Example**: `C:\Users\Manya\Documents\Projects\data\employees.csv`
*   **Linux/macOS Example**: `/home/manya/Documents/Projects/data/employees.csv`

### Key Characteristic
An absolute path **always points to the exact same file**, regardless of where your Python script or Jupyter Notebook is currently running on the machine. It is like using exact GPS coordinates.

---

## 2. Relative Paths (The Directional Pointer)
A **Relative Path** specifies a location relative to your program's **Current Working Directory (CWD)**—the folder where your Python script is currently executing.

*   **Examples**: 
    *   `employees.csv` (Look right inside the current folder)
    *   `data/employees.csv` (Look inside a folder named 'data' that sits in the current folder)
    *   `reports/output.csv`

### Key Characteristic
These paths **do not** begin from the root of the file system. Instead, Python locates the current working directory first, and then appends the relative path to find the target file. If you move your script to a different folder, relative paths will break unless the surrounding folder structure stays exactly the same.


# Pathlib Internal Working: How Python Resolves Relative Paths

When you use relative paths, Python doesn't guess where files are. It uses a strict mathematical combination system behind the scenes to map out the exact hard drive coordinates.

---

## The Internal Concatenation Flow

Suppose your current working directory (**CWD**) on your machine is:
`C:\Users\Manya\Projects`

Now, you write this line of code inside your script:
```python
path = Path("employees.csv")
```

Here is exactly how Python processes that relative path internally to find the file:

```text
       Current Working Directory (CWD)
         [ C:\Users\Manya\Projects ]
                      │
                      ▼
               Relative Input
              [ employees.csv ]
                      │
                      ▼
         Internal Path Concatenation
    [ C:\Users\Manya\Projects\employees.csv ]
```

### Why This Matters
Notice that **you did not type the full address**, but Python successfully determined the exact file location. It achieves this by automatically anchoring any relative path string directly to the root of your active runtime environment.


In [3]:
from pathlib import Path

path = Path("employees.csv")

print(path) ## this is the relative path

employees.csv


# Current Working Directory (CWD)

The Current Working Directory (CWD) is the directory from which Python is currently executing your program.

Every relative path is interpreted relative to the CWD.

In [4]:
from pathlib import Path

print(Path.cwd())

E:\python-for-data-engineering\Professional Python


# Home Directory

Every user has a home directory.

Examples

Windows

```text
C:\Users\Manya
```

Linux

```text
/home/manya
```

Python provides `Path.home()` to access it.

In [5]:
from pathlib import Path

print(Path.home())

C:\Users\manya


# Dynamically Finding the User Space: `Path.home()`

In professional software development and automated data pipelines, applications frequently need to read or write user-specific data inside default system locations such as:
*   `Downloads` (For processing incoming files)
*   `Documents` (For saving local database files)
*   `Desktop` / `Pictures`
*   `.config` / App settings directories

---

## The Hardcoding Anti-Pattern
If you hardcode a direct string path to your user folder, your code becomes fragile and will break instantly on any other computer:

```python
# ❌ Hardcoded: Breaks if the user's name is not 'Manya', or on Mac/Linux
my_path = "C:\\Users\\Manya\\Downloads"
```

If a colleague named *Rahul* runs this script, or if the code is deployed to a cloud server, the program will crash with a `FileNotFoundError` because the directory `C:\Users\Manya` does not exist on their machine.

---

## The `Path.home()` Solution
The `Path.home()` method acts as a dynamic system scanner. It asks the operating system, *"Who is currently logged in, and where is their profile folder?"* and returns that exact absolute path object instantly.

```python
from pathlib import Path

# ✅ Universal Standard: Works for any username, on any Operating System
download_folder = Path.home() / "Downloads"
```

### Why This is Essential for Data Engineers
1.  **Zero Configuration**: The exact same script can be shared with a team of 50 developers without requiring anyone to change hardcoded strings or environment configurations.
2.  **Clean Chaining**: It returns a true `Path` object, allowing you to instantly use the `/` operator to drill down into deeper sub-folders seamlessly.


# resolve()

`resolve()` converts a path into its absolute form.

Syntax

```python
path.resolve()
```

In [6]:
from pathlib import Path

path = Path("employees.csv")

print(path.resolve())

E:\python-for-data-engineering\Professional Python\employees.csv


In [7]:
from pathlib import Path

path = Path("data/report.csv")

print("Relative :", path)

print("Absolute :", path.resolve())

Relative : data\report.csv
Absolute : E:\python-for-data-engineering\Professional Python\data\report.csv


## Does resolve() Create the File?

No.

- This is a common misconception.

- resolve() only calculates the absolute location of the path.

- It does not create the file or directory.

# Why Professional Projects Prefer Relative Paths

In enterprise software engineering and data pipelining, using **Relative Paths** within a project repository is a mandatory best practice. 

---

## The Production Problem: Hardcoded Absolute Paths
Imagine you are building a tool named `EngineeringTool`. If you use an absolute path to reference a dataset inside your project folder, your code looks like this:

```python
# ❌ Fragile Architecture: Locked to one specific machine
data_path = Path("C:/Users/Manya/Documents/EngineeringTool/input/employees.csv")
```

If you push this code to GitHub, GitLab, or a shared corporate repository, it becomes completely unusable for anyone else. The moment another engineer pulls the repository down, or a cloud server tries to run it, the script will crash because your personal user directory (`C:/Users/Manya/...`) does not exist on their system.

---

## The Structural Standard: Relative Paths
By using relative paths, you anchor your file searches to the **Project Root Directory** instead of the computer's hard drive root.

### Target Project Architecture
```text
EngineeringTool/
├── config/
│   └── config.json
├── input/
│   └── employees.csv
└── reports/
```

Instead of mapping the whole machine layout, you specify directions starting from inside `EngineeringTool/`:

```python
# ✅ Highly Portable Standard: Works everywhere
data_path = Path("input") / "employees.csv"
```

### Strategic Benefits for Production Environments
*   **Instant Portability**: If a teammate clones your project repo onto a Mac, Windows, or Linux laptop, the path works out of the box. The surrounding computer architecture does not matter as long as the internal project folder structure remains intact.
*   **Containerization & CI/CD Ready**: Automated testing pipelines and Docker containers isolate code into temporary running directories. Relative paths ensure your scripts navigate these virtual execution folders seamlessly without throwing path errors.


# Path Properties

A `Path` object stores much more than just a string.

It knows information about the file and directory it represents.

Some of the most commonly used properties are:

- name
- stem
- suffix
- suffixes
- parent
- parents
- parts
- drive
- anchor

These properties allow us to retrieve information about a path without manually splitting strings.

In [8]:
from pathlib import Path

path = Path("data/reports/employee_report.csv")

print(path.name) # The name property returns the last component of the path.

employee_report.csv


# Pathlib Properties & File Renaming

## Key Concepts
*   **`path.stem`**: Extracts the **core filename** only, stripping away the extension.
*   **`path.suffix`**: Extracts the **file extension** only (includes the dot, e.g., `.csv`).
*   **The Anti-Pattern**: Do NOT use `.replace(".csv", "")` or string slicing to drop extensions. It breaks on files like `data.backup.csv`.
*   **The Standard**: Use `path.stem + "_suffix.csv"` for flawless, safe file transformations.

## Core Properties At A Glance
For a file path like `Path("data/input/employees.csv")`:
*   **`path.name`** ➔ `employees.csv` (Full filename)
*   **`path.stem`** ➔ `employees` (Filename without extension)
*   **`path.suffix`** ➔ `.csv` (Extension only)
*   **`path.parent`** ➔ `data/input` (Containing folder)


In [9]:
from pathlib import Path

path = Path("employee_report.csv")

print(path.stem) # The stem is the filename without its final extension.

employee_report


In [11]:
from pathlib import Path

path = Path("employee_report.csv")

print(path.suffix) # Returns the last file extension.

.csv


## Suffixes
- Some files have multiple extensions.
- The suffixes property returns all extensions.

In [12]:
from pathlib import Path

path = Path("backup.tar.gz")

print(path.suffixes)

['.tar', '.gz']


In [14]:
from pathlib import Path

path = Path("sales_2026.xlsx")

print(path.name)
print(path.stem)
print(path.suffix)

sales_2026.xlsx
sales_2026
.xlsx


In [10]:
from pathlib import Path

# --- Quick Revision Playground ---
file_path = Path("analytics/logs/system_report.json")

# 1. Inspect Properties directly
print(f"File Name: {file_path.name}")
print(f"Core Stem: {file_path.stem}")
print(f"Extension: {file_path.suffix}")

# 2. Re-naming Recipe
new_output = file_path.parent / f"{file_path.stem}_summary.csv"
print(f"New Output: {new_output}")


File Name: system_report.json
Core Stem: system_report
Extension: .json
New Output: analytics\logs\system_report_summary.csv


## The `.parent` Property

## Key Concepts
*   **Definition**: `path.parent` isolates the **directory immediately containing** the file or folder.
*   **The Blueprint**: It works by evaluating the path structure and dropping the very last element.
*   **Best Use Case**: Saving new files (like logs, summaries, or backups) in the exact same directory as your input file.

## Internal Mechanics
```text
data ➔ reports ➔ employee_report.csv  ➔ [Drop Last Component] ➔ data/reports
```


In [17]:
from pathlib import Path

config_path = Path("data/reports/employee_report.csv")

# 1. Isolate the containing directory
containing_folder = config_path.parent
print(f"Target Parent Directory: {containing_folder}")

# 2. Dynamic Asset Creation Recipe
# Instantly generate a sibling log file path inside the exact same folder
log_file = config_path.parent / "application.log"
print(f"Generated Sibling Log:  {log_file}")


Target Parent Directory: data\reports
Generated Sibling Log:  data\reports\application.log


# The Path Combining Operator (`/`)

## Key Concepts
*   **The Operator**: `pathlib` overrides Python's division operator (`/`) to act as a **universal path joiner**.
*   **Syntax Rules**: At least one of the items must be an active `Path` object. You can chain text strings to it seamlessly afterward.
*   **Cross-Platform Translation**: It replaces old string joining tricks (`+ "\\"`) completely. `pathlib` takes care of converting the slashes behind the scenes to fit the machine's operating system environment.

## Execution Model
```text
Path("reports")  ➔  /  ➔  "summary.csv"  ➔  reports/summary.csv (Linux/Mac)
                                         ➔  reports\summary.csv (Windows)
```


In [18]:
from pathlib import Path

# 1. Initialize base folder object
folder = Path("reports")

# 2. Combine paths cleanly using the / operator
target_file = folder / "summary.csv"
print(f"Combined Result: {target_file}")

# 3. Multi-level chaining demonstration
nested_file = Path("data") / "logs" / "archive" / "system.log"
print(f"Nested Result:   {nested_file}")


Combined Result: reports\summary.csv
Nested Result:   data\logs\archive\system.log


# The `.parents` Property (All Ancestor Directories)

## Key Concepts
*   **Definition**: `path.parents` returns a sequence containing **all parent folders** going up the directory tree to the root.
*   **Sequence Access**: Because it returns an immutable sequence of Path objects, individual ancestor directories are accessed using standard zero-based indexing (`[0]`, `[1]`, `[2]`).
*   **Index Mapping**: 
    *   `[0]` is the immediate parent folder (identical to calling `.parent`).
    *   `[1]` is the grandparent folder (one level higher).
    *   `[2]` is the great-grandparent folder, and so on.

## Ancestor Mapping Engine
```text
  [2] data  ➔  [1] reports  ➔  [0] 2026  ➔  employee_report.csv
```


In [19]:
from pathlib import Path

# Initialize a multi-layered nested path
path = Path("data/reports/2026/employee_report.csv")

print(path.parents)

# 1. Access individual ancestor levels via index
print(f"Immediate Parent: {path.parents[0]}")
print(f"Grandparent: {path.parents[1]}")
print(f"Great-Grandparent: {path.parents[2]}")

print("\n--- Iterating Through All Ancestors ---")
# 2. Loop through the sequence to view the entire hierarchy
for level, ancestor in enumerate(path.parents):
    print(f"Level {level}: {ancestor}")


<WindowsPath.parents>
Immediate Parent: data\reports\2026
Grandparent: data\reports
Great-Grandparent: data

--- Iterating Through All Ancestors ---
Level 0: data\reports\2026
Level 1: data\reports
Level 2: data
Level 3: .


## parts

- Returns every component of the path as a tuple.

In [20]:
from pathlib import Path

path = Path("data/reports/employee_report.csv")

print(path.parts)

('data', 'reports', 'employee_report.csv')


# The `.drive` Property

## Key Concepts
*   **Definition**: `path.drive` extracts the **root drive letter** from a path string.
*   **Windows Environment**: It returns the drive letter followed by a colon (e.g., `C:` or `D:`).
*   **Linux / macOS Environment**: It returns an empty string (`""`) because POSIX systems mount files under a single root directory (`/`) instead of using separate drive letters.
*   **Best Use Case**: Adding safety checks in data workflows to confirm your pipeline is reading from the correct storage drive before processing massive files.

## Behavior Model
```text
Windows:  C:/Users/Manya/report.csv  ➔ .drive ➔ "C:"
Linux:    /home/manya/report.csv     ➔ .drive ➔ ""
```


In [21]:
from pathlib import Path

# --- Windows Path Simulation ---
win_path = Path("C:/Users/Manya/report.csv")
print(f"Windows Drive: '{win_path.drive}'")

# --- Linux/macOS Path Simulation ---
linux_path = Path("/home/manya/report.csv")
print(f"Linux Drive:   '{linux_path.drive}' (Empty string)")


Windows Drive: 'C:'
Linux Drive:   '' (Empty string)


# The `.anchor` Property

## Key Concepts
*   **Definition**: `path.anchor` combines the **drive component** and the **root directory separator** to return the absolute start of a path.
*   **Windows Architecture**: Returns the drive letter with the trailing backslash separator (e.g., `C:\`).
*   **Linux / macOS Architecture**: Returns a single forward slash root symbol (`/`) because there are no drive letters.
*   **Best Use Case**: Detecting whether a path object is anchored to the system root (making it an absolute path) or if it is unanchored (making it a relative path).

## Behavior Model
```text
Windows:  C:/Users/Manya/report.csv  ➔ .anchor ➔ "C:\\"
Linux:    /home/manya/report.csv     ➔ .anchor ➔ "/"
```


In [22]:
from pathlib import Path

# --- Windows Path Simulation ---
win_path = Path("C:/Users/Manya/report.csv")
print(f"Windows Anchor: '{win_path.anchor}'")

# --- Linux/macOS Path Simulation ---
linux_path = Path("/home/manya/report.csv")
print(f"Linux Anchor:   '{linux_path.anchor}'")


Windows Anchor: 'C:\'
Linux Anchor:   '\'


# File Pattern Searching with `.glob()`

## Key Concepts
*   **Definition**: The `.glob()` method searches a directory for files and folders that match a specific wildcard pattern (like `*.csv` or `*.json`).
*   **The Generator Mechanism**: It does **not** return a static list. Instead, it returns a **generator object** (an iterator) that lazily yields matching items one by one. This keeps memory usage low when scanning folders with millions of files.
*   **Smart Objects**: Every item yielded by the generator is a fully functioning `Path` object (e.g., `WindowsPath` or `PosixPath`), not a plain string. 
*   **Zero Conversion**: Because the items are already `Path` objects, you can immediately access their attributes (`.name`, `.stem`, `.suffix`) inside your loop without wrapping them in another function.

## Wildcard Pattern Reference
*   `*.csv` ➔ Finds all files ending with `.csv` in the immediate folder.
*   `*`     ➔ Finds every single file and folder in the immediate directory.


In [24]:
from pathlib import Path

# 1. Define and create the mock project directory safely
folder = Path("project")
folder.mkdir(exist_ok=True)  # exist_ok=True prevents errors if the folder already exists

# 2. Populate the folder with dummy testing files using .touch()
test_files = [
    "employees.csv",
    "commits.csv",
    "builds.csv",
    "report.json",
    "config.json",
    "notes.txt"
]

for file_name in test_files:
    file_target = folder / file_name
    file_target.touch()  # Creates an empty file on your disk

print("✨ Mock environment setup complete! Files generated inside 'project/' folder.")
print("----------------------------------------------------------------------")

# 3. Now execute your .glob() query on the real files
csv_search_results = folder.glob("*.csv")
print(f"Return Type: {type(csv_search_results)}") 
print("*(Notice it is a generator object)*\n")

print("--- Iterating Through Matching Path Objects ---")
for file in folder.glob("*.csv"):
    print(f"Found Object: {file}")
    print(f"  -> File Name: {file.name}")
    print(f"  -> Core Stem: {file.stem}")
    print(f"  -> Extension: {file.suffix}\n")


✨ Mock environment setup complete! Files generated inside 'project/' folder.
----------------------------------------------------------------------
Return Type: <class 'map'>
*(Notice it is a generator object)*

--- Iterating Through Matching Path Objects ---
Found Object: project\builds.csv
  -> File Name: builds.csv
  -> Core Stem: builds
  -> Extension: .csv

Found Object: project\commits.csv
  -> File Name: commits.csv
  -> Core Stem: commits
  -> Extension: .csv

Found Object: project\employees.csv
  -> File Name: employees.csv
  -> Core Stem: employees
  -> Extension: .csv



# Checking Files and Directories

Before reading or writing anything, we should verify what the given path represents.

The most commonly used methods are:

- exists()
- is_file()
- is_dir()

These methods help prevent runtime errors and make programs more robust.

## exits()
- checks whether a file or directory actually exists in the file system.

In [25]:
from pathlib import Path

path = Path("employees.csv")

print(path.exists())

False


# Lazy Evaluation and Defensive File Handling

## Key Concepts
*   **Lazy Evaluation**: Instantiating a `Path` object like `Path("salary.csv")` **never** checks the hard drive. It only stores the path coordinates as a blueprint in memory, meaning it will never throw an error on creation even if the file is completely missing.
*   **The Error Trigger**: The error only triggers when you attempt a physical disk operation—like reading, writing, or opening the file (`path.open()`).
*   **Defensive Programming**: To prevent your pipelines from crashing with a `FileNotFoundError`, use the `.exists()` method to verify the file is physically present before interacting with its contents.

## Execution Safeguard Model
```text
Path("missing.csv") ➔ Memory Blueprint Only (Safe)
       │
       ├── ❌ path.open()   ➔ Triggers FileNotFoundError
       └──  path.exists() ➔ Returns False (Safe Check)
```


In [28]:
from pathlib import Path

# 1. Define a file that definitely does not exist on your disk
non_existent_file = Path("salary_2026_missing.csv")

print("==================================================")
print("          DEFENSIVE PATH HANDLING TEST            ")
print("==================================================")
print("👉 Step 1: Path object instantiated successfully without error.")
print(f"   Stored Coordinate Mapping: {non_existent_file}\n")

# 2. Safe verification check using .exists()
if non_existent_file.exists():
    print("👉 Step 2: Processing file...")
    with non_existent_file.open("r") as f:
        print(f.read())
else:
    print("👉 Step 2: [Safe Bypass] File not found on disk. Skipping to avoid crashing.")
print("==================================================")

# 3. Code that would crash the pipeline if run un-safeguarded:
# non_existent_file.open() # Uncommenting this line will raise a FileNotFoundError!


          DEFENSIVE PATH HANDLING TEST            
👉 Step 1: Path object instantiated successfully without error.
   Stored Coordinate Mapping: salary_2026_missing.csv

👉 Step 2: [Safe Bypass] File not found on disk. Skipping to avoid crashing.


# File Type Verification using `.is_file()`

## Key Concepts
*   **The Problem**: A path can point to a location that exists on your system, but it might be a folder (directory) instead of an actual file.
*   **The Method**: `path.is_file()` validates the specific type of path. It returns `True` only if the target points directly to a file. It returns `False` if the target is a folder or does not exist.
*   **Production Safety**: This method is used in data pipelines to ensure you do not accidently try to open or parse a folder when searching for data logs or reports.

## Behavior Summary
*   `Path("employees.csv").is_file()` ➔ `True` (Target is a file)
*   `Path("reports_folder").is_file()` ➔ `False` (Target is a directory)


In [29]:
from pathlib import Path

# Create a mock environment with one file and one folder
Path("employees.csv").touch()
Path("reports").mkdir(exist_ok=True)

# 1. Test a true file path
file_path = Path("employees.csv")
print(f"Path: {file_path}")
print(f"Is it a file? {file_path.is_file()}\n")

# 2. Test a directory path
folder_path = Path("reports")
print(f"Path: {folder_path}")
print(f"Is it a file? {folder_path.is_file()}")


Path: employees.csv
Is it a file? True

Path: reports
Is it a file? False


# Directory Verification using `.is_dir()`

## Key Concepts
*   **The Method**: `path.is_dir()` checks if a specific path points to a directory (folder). It returns `True` if the path exists and is a folder, and `False` if it is a file or does not exist.
*   **Production Safety**: This method is used when automation scripts need to save files into an export directory, ensuring the destination is a valid folder before running write operations.

## Behavior Summary
*   `Path("reports").is_dir()`      ➔ `True` (Target is a directory)
*   `Path("employees.csv").is_dir()` ➔ `False` (Target is a file)


In [30]:
from pathlib import Path

# Ensure the mock file and folder exist from the previous step
Path("reports").mkdir(exist_ok=True)
Path("employees.csv").touch()

# 1. Test a directory path
folder_path = Path("reports")
print(f"Path: {folder_path}")
print(f"Is it a directory? {folder_path.is_dir()}\n")

# 2. Test a file path
file_path = Path("employees.csv")
print(f"Path: {file_path}")
print(f"Is it a directory? {file_path.is_dir()}")


Path: reports
Is it a directory? True

Path: employees.csv
Is it a directory? False


| Method      | Checks               |
| ----------- | -------------------- |
| `exists()`  | Does the path exist? |
| `is_file()` | Is it a file?        |
| `is_dir()`  | Is it a directory?   |


## Example

In [32]:
from pathlib import Path

input_file = Path("employees.csv")

if input_file.exists() and input_file.is_file():
    with input_file.open("r") as file:
        # Process the file
        pass
else:
    print("Input file not found.")

# Creating Directories and Files

One of the most common tasks in automation and ETL pipelines is creating folders and files.

The primary methods are:

- mkdir()
- touch()

These allow your program to prepare the file system before processing data.

# Creating Folders using `.mkdir()`

## Key Concepts
*   **Definition**: `mkdir()` stands for **Make Directory**. It physically creates a new folder on your hard drive at the specified path location.
*   **The Syntax**: `path.mkdir()` creates the final folder in the path sequence.
*   **Default Behavior**: If the folder already exists, Python will throw a `FileExistsError`. If any of the parent folders in the path are missing, it will throw a `FileNotFoundError`.

## Structural View
```text
Before: Project/ ➔ input/, config/
After:  Project/ ➔ input/, config/, reports/ (New)
```
- mkdir() doesn't create a Python object.

- It asks the Operating System to create an actual folder.

In [33]:
from pathlib import Path

# 1. Define the new folder target location
new_folder = Path("reports")

# 2. Check if it exists, then create it safely
if not new_folder.exists():
    new_folder.mkdir()
    print("Folder 'reports' created successfully.")
else:
    print("Folder already exists.")


Folder already exists.


# Handling Existing Folders with `.mkdir()`

## Key Concepts
*   **The Conflict**: If you run `path.mkdir()` on a directory that already exists, Python raises a `FileExistsError`.
*   **The Reason**: This safety feature prevents scripts from accidentally overwriting, altering, or conflicting with existing folders during automated runs.
*   **Explicit Action**: Python requires you to intentionally specify how to handle folders that are already present on the storage drive.

## Behavior Summary
```text
folder.mkdir() ➔ Directory exists ➔ Raises FileExistsError
```


In [34]:
from pathlib import Path

# Setup: Ensure the folder exists
test_folder = Path("reports")
test_folder.mkdir(exist_ok=True)

# 1. Triggering the error explicitly to see the default behavior
try:
    test_folder.mkdir()
except FileExistsError as e:
    print(f"Caught expected error: {type(e).__name__} - Folder already exists!")


Caught expected error: FileExistsError - Folder already exists!


# Safe Directory Creation using `exist_ok=True`

## Key Concepts
*   **The Parameter**: Adding `exist_ok=True` modifies the default behavior of `.mkdir()`. It allows the script to continue without raising a `FileExistsError` if the directory is already present.
*   **Enterprise Standard**: This is a mandatory best practice in data pipelines. It ensures that automated scripts can run repeatedly without crashing on infrastructure that was already set up during a previous run.

## Internal Mechanics Comparison
```text
Without exist_ok=True: Directory Exists? ➔ Yes ➔ Raise FileExistsError (Crash)
With exist_ok=True:    Directory Exists? ➔ Yes ➔ Do Nothing (Continue Program)
```


In [35]:
from pathlib import Path

# Define the folder target
folder = Path("reports")

# 1. First run: Creates the folder if it does not exist
folder.mkdir(exist_ok=True)
print("First check passed (Created or verified).")

# 2. Second run: Folder definitely exists, but runs safely without errors
folder.mkdir(exist_ok=True)
print("Second check passed (Safely ignored existing folder).")


First check passed (Created or verified).
Second check passed (Safely ignored existing folder).


# Nested Directory Creation using `parents=True`

## Key Concepts
*   **The Problem**: If you try to create a deeply nested folder path (like `reports/2026/July/Week1`) using a standard `.mkdir()`, the operating system throws a `FileNotFoundError` if any of the intermediate parent directories are missing.
*   **The Parameter**: Adding `parents=True` tells Python to look down the entire path chain and automatically create every single missing parent folder required to reach the final directory.
*   **The Enterprise Standard**: Combining both options (`parents=True, exist_ok=True`) creates a completely error-proof directory setup statement that safely builds missing infrastructure on any machine without crashing if elements already exist.

## Internal Mechanics Comparison
```text
Without parents=True: Check target ➔ Parent missing ➔ Raise FileNotFoundError
With parents=True:    Create reports ➔ Create 2026 ➔ Create July ➔ Create Week1
```


In [36]:
from pathlib import Path

# 1. Define a deeply nested corporate reporting path
nested_target = Path("output/2026/July/DailyReports")

# 2. Deploy the standard enterprise directory creation recipe
nested_target.mkdir(parents=True, exist_ok=True)

print(f"✨ Directory structure verified and ready at:\n👉 {nested_target}")
print(f"Directory physically exists? {nested_target.is_dir()}")


✨ Directory structure verified and ready at:
👉 output\2026\July\DailyReports
Directory physically exists? True


# Empty File Creation using `.touch()`

## Key Concepts
*   **Definition**: The `.touch()` method physically creates a blank, empty file on your storage system at the path specified.
*   **Default Behavior**: If the file does not exist, the operating system generates a new blank file. If the file already exists, it remains completely unchanged (no error is raised, and your data is not overwritten).
*   **Contrast with `.mkdir()`**: Unlike `.mkdir()`, which crashes on existing paths unless configured otherwise, `.touch()` is inherently safe to execute repeatedly without extra parameters.

## Execution Model
```text
Path("application.log") ➔ .touch() ➔ Operating System ➔ Generates Empty File
```


In [37]:
from pathlib import Path

# 1. Define the target file name
log = Path("application.log")

# 2. Generate the file on disk
log.touch()

# 3. Verify the file's presence and type
print(f"Does the file exist? {log.exists()}")
print(f"Is it confirmed as a file? {log.is_file()}")


Does the file exist? True
Is it confirmed as a file? True


## The Directory Pre-requisite Rule
While `.touch()` handles existing files gracefully, it will **fail completely** with a `FileNotFoundError` if you try to create a file inside a folder path that does not physically exist yet. 

To prevent production crashes, you must always ensure the containing directory structure is fully built out *before* running the file creation step.


In [38]:
from pathlib import Path

# Define a path pointing inside a potentially missing directory
log_target = Path("logs/app.log")

# 1. First step: Safely build out the containing parent folder network
log_target.parent.mkdir(parents=True, exist_ok=True)

# 2. Second step: Safely generate the empty file inside that confirmed folder
log_target.touch()

print(f"✨ Safe deployment pattern successful. File created at: {log_target}")


✨ Safe deployment pattern successful. File created at: logs\app.log


## Production Staging Workflows
Data pipelines commonly run a standard staging pattern to guarantee that both the destination folder and the target data file infrastructure are completely prepared before any extraction or writing scripts attempt to stream production data.


In [39]:
from pathlib import Path

# 1. Isolate and build the output directory staging area
output_folder = Path("output")
output_folder.mkdir(exist_ok=True)

# 2. Formulate the absolute target file location
report = output_folder / "summary.csv"

# 3. Secure the empty file placeholder on disk
report.touch()

print(f"🚀 Staging environment successfully prepared for target: {report}")


🚀 Staging environment successfully prepared for target: output\summary.csv


| Method                 | Purpose                           |
| ---------------------- | --------------------------------- |
| `mkdir()`              | Create a directory                |
| `mkdir(exist_ok=True)` | Don't fail if directory exists    |
| `mkdir(parents=True)`  | Create missing parent directories |
| `touch()`              | Create an empty file              |


# Reading and Writing Files with pathlib

The `Path` object provides built-in methods for reading and writing files.

Common methods:

- open()
- read_text()
- write_text()
- read_bytes()
- write_bytes()

These methods make file operations cleaner and more object-oriented.

| Method          | Returns                      | Purpose                   |
| --------------- | ---------------------------- | ------------------------- |
| `open()`        | File Object                  | Traditional file handling |
| `read_text()`   | String                       | Read complete text file   |
| `write_text()`  | Number of characters written | Write text file           |
| `read_bytes()`  | Bytes                        | Read binary file          |
| `write_bytes()` | Number of bytes written      | Write binary file         |


# File Operations: Reading and Writing Data with `pathlib`

## 1. The `.open()` Method
The `path.open()` method acts exactly like Python's built-in `open()` function but eliminates the need to convert path objects into strings.

*   **Syntax**: `path.open(mode="r", encoding="utf-8")`
*   **Mechanism**: It delegates the request directly to the operating system's file descriptor system to stream file contents.
*   **Comparison**:
    *   *Traditional*: `with open(str(path)) as file:`
    *   *Modern*: `with path.open() as file:`


In [40]:
from pathlib import Path

# Setup dummy testing text file
path = Path("employees.txt")
path.write_text("John\nAlice\nBob")

# Open and read the file contents using .open()
with path.open("r") as file:
    print(file.read())


John
Alice
Bob


## 2. The `.read_text()` and `.write_text()` Shortcuts
These are high-level convenience methods designed to eliminate boilerplate code for small text files.

*   **`path.read_text()`**: Automatically opens the file, reads the entire string into memory, and closes the file in a single step.
*   **`path.write_text(data)`**: Automatically opens the file in write mode (`"w"`), commits the text string, and closes the file. **Warning**: This overwrites any pre-existing file content completely.
*   **Usage Rule**: Perfect for small files like JSON, configs, or logs. Do **not** use them for massive data sets, as they load the entire file into RAM at once.


In [41]:
from pathlib import Path

report = Path("report.txt")

# 1. Write content automatically without a 'with' block
report.write_text("Weekly Engineering Report")

# 2. Read content automatically in a single line
text_data = report.read_text()
print(f"File Contents: {text_data}")


File Contents: Weekly Engineering Report


## 3. Binary Operations: `.read_bytes()` and `.write_bytes()`
When dealing with non-text assets (images, PDFs, ZIP archives, or compiled files), use byte-level storage methods.

*   **`path.read_bytes()`**: Streams raw un-decoded data directly into a Python `bytes` object.
*   **`path.write_bytes(b_data)`**: Commits raw binary sequences directly onto the physical storage disk.


In [42]:
from pathlib import Path

binary_file = Path("sample.bin")

# 1. Write raw binary data using a bytes literal (b"...")
binary_file.write_bytes(b"ABC123")

# 2. Read raw binary bytes back from the disk
raw_bytes = binary_file.read_bytes()
print(f"Data Type: {type(raw_bytes)}")
print(f"Raw Bytes: {raw_bytes}")


Data Type: <class 'bytes'>
Raw Bytes: b'ABC123'


## 4. Production Choice: Traditional `open()` vs. `pathlib` Shortcuts

### When to use `.read_text()` / `.write_text()`
*   Quickly reading configurations or injection scripts (`config.json`, `query.sql`).
*   Dumping clean, processed system string summaries into an archive folder.

### When to use `path.open()`
*   Processing very large datasets line-by-line (`for line in file:`).
*   Appending new data entries seamlessly to an existing file using append mode (`mode="a"`).
*   Handling memory-safe chunk tracking or streaming complex multi-part media pipelines.


In [43]:
from pathlib import Path

# Professional Defensive Pipeline Template
config_file = Path("config.json")

# Seed the file for testing purposes
config_file.write_text('{"status": "active"}')

# Safe checking and handling wrapper
if config_file.exists():
    pipeline_settings = config_file.read_text()
    print(f"Pipeline Config Loaded Successfully: {pipeline_settings}")
else:
    raise FileNotFoundError("Critical configuration asset missing.")


Pipeline Config Loaded Successfully: {"status": "active"}


# Searching Files and Directories

One of the biggest advantages of `pathlib` is its ability to search files using patterns.

The three most commonly used methods are:

- iterdir()
- glob()
- rglob()

These methods return `Path` objects, allowing you to immediately work with the matching files and directories.

# Comprehensive Search Environment Setup

To master file searching, filtering, and directory traversal techniques, we need a realistic, multi-layered mock folder structure. This environment includes various file formats nested across different subdirectories.

## Targeted Project Topology
```text
Project/
│
├── employees.csv
├── commits.csv
├── report.json
├── config.json
├── notes.txt
├── builds.xlsx
│
├── reports/
│   ├── weekly.csv
│   ├── monthly.csv
│   └── yearly.xlsx
│
└── archive/
    └── old.csv
```


In [44]:
from pathlib import Path

# 1. Establish structural path parameters
base_dir = Path("Project")
reports_dir = base_dir / "reports"
archive_dir = base_dir / "archive"

# 2. Build the directory layout safely
reports_dir.mkdir(parents=True, exist_ok=True)
archive_dir.mkdir(parents=True, exist_ok=True)

# 3. Define all standard file targets
target_files = [
    base_dir / "employees.csv",
    base_dir / "commits.csv",
    base_dir / "report.json",
    base_dir / "config.json",
    base_dir / "notes.txt",
    base_dir / "builds.xlsx",
    reports_dir / "weekly.csv",
    reports_dir / "monthly.csv",
    reports_dir / "yearly.xlsx",
    archive_dir / "old.csv"
]

# 4. Generate all mock files on disk
for file_item in target_files:
    file_item.touch()

print("✨ Search validation topology successfully mapped onto disk structure.")


✨ Search validation topology successfully mapped onto disk structure.


## iterdir()

- This is the simplest searching method.

- It returns everything immediately inside a directory.

Syntax
```python
folder.iterdir()
```

In [45]:
from pathlib import Path

folder = Path("Project")

for item in folder.iterdir():
    print(item)

Project\archive
Project\builds.csv
Project\builds.xlsx
Project\commits.csv
Project\config.json
Project\employees.csv
Project\notes.txt
Project\report.json
Project\reports


# Managing Files and Directories

The `pathlib` module also provides methods to rename, move, and delete files or directories.

The most commonly used methods are:

- rename()
- replace()
- unlink()
- rmdir()

These methods modify the file system, so they should be used carefully.

## rename()

- rename() changes the name or location of a file or directory.

Syntax
```python
path.rename(target)
```

- where target is another Path object or a path string.

In [ ]:
from pathlib import Path

old = Path("report.csv")

old.touch() # it will create the file on disk itself

new = Path("weekly_report.csv")

old.rename(new)

Path

↓
employees.csv

↓

rename()

↓

Operating System

↓

Rename File

↓

employees_list.csv

In [10]:
## Move a file into another folder.

In [12]:
from pathlib import Path

source = Path("weekly_report.csv")

destination = Path("reports/weekly_report.csv")

source.rename(destination)

WindowsPath('reports/weekly_report.csv')

## Important Note

- rename() does not create missing directories.

- If the archive folder doesn't exist,
```python
source.rename(destination)
```

- raises a FileNotFoundError.

- Always ensure the destination directory exists first.

## replace()

- replace() is similar to rename(), but it will replace the destination if it already exists (subject to platform behavior).

In [14]:
from pathlib import Path

new_report = Path("report.txt")

new_report.replace("output/summary.csv")

WindowsPath('output/summary.csv')

| Method      | Destination Exists                       |
| ----------- | ---------------------------------------- |
| `rename()`  | Behavior depends on the operating system |
| `replace()` | Intended to replace the destination      |


## unlink()

- unlink() deletes a file.

Syntax
```python
path.unlink()
```

In [17]:
from pathlib import Path

temp = Path("temporary.csv")

temp.touch()

temp.unlink()

## Important

- unlink() only deletes files.

Trying
```python
Path("reports").unlink()
```
- where reports is a directory raises an error.

In [18]:
## This avoids attempting to delete a file that doesn't exist.

from pathlib import Path

temp = Path("temporary.csv")

if temp.exists():

    temp.unlink()

## rmdir()

- rmdir() removes an empty directory.

Syntax
```python
path.rmdir()
```

## What if the directory contains files?
- raises OSError Directory not empty

- Python prevents accidental deletion of folders containing data.

In [19]:
from pathlib import Path

base_dir=Path("reports")
empty_dir= base_dir/ Path("empty") 

empty_dir.mkdir(parents=True,exist_ok=True)## created the empty directory

In [23]:
## Now we will delete the empty directory
folder = Path("reports/empty")

folder.rmdir()

| Method      | Purpose                         |
| ----------- | ------------------------------- |
| `rename()`  | Rename or move a file/directory |
| `replace()` | Replace the destination file    |
| `unlink()`  | Delete a file                   |
| `rmdir()`   | Delete an empty directory       |


## Enterprise Example

A typical **ETL pipeline**:

`Input` → `Process CSV` → `Generate Report` → `Move Processed File` → `Delete Temporary File`

```python
from pathlib import Path

temp = Path("temp.csv")
if temp.exists():
    temp.unlink()
```


## Another Example

Archive **processed reports**.

```python
from pathlib import Path

report = Path("report.csv")
archive = Path("archive")
archive.mkdir(exist_ok=True)
report.rename(archive / report.name)
```

Notice the reuse of **`report.name`**. If the file is `report.csv`, it becomes `archive/report.csv` without manually writing the filename.


## Best Practices

* **File Deletion**: Check that a file exists before deleting it.
  ```python
  if file.exists():
      file.unlink()
  ```

* **Directory Creation**: Create destination directories before moving files.
  ```python
  destination.parent.mkdir(parents=True, exist_ok=True)
  ```

* **Path Operations**: Use Path operations instead of string concatenation.
  ```python
  archive / report.name  # Do this
  # "archive/" + report.name  # Avoid this
  ```
